# Part 2: DataFrames Under the Hood — Pandas vs Polars

**Goal:** Understand how two dataframe libraries store and process data differently, and see how those design decisions affect performance on your dataset.

## 1. Row-Oriented vs Column-Oriented Storage

Before comparing Pandas and Polars, you need to understand a concept from database design: **row-oriented vs column-oriented storage**.

**Row-oriented** stores all fields of one record together, then the next record, etc:
```
[timestamp₁, sensor_A₁, label₁] [timestamp₂, sensor_A₂, label₂] ...
```

**Column-oriented** stores all values of one field together, then the next field, etc:
```
[timestamp₁, timestamp₂, ...] [sensor_A₁, sensor_A₂, ...] [label₁, label₂, ...]
```

Row storage is faster when you need entire records (looking up one patient's chart). Column storage is faster when you need to scan one field across many records (average all patients' blood pressure). This is the tradeoff that separates transactional databases like PostgreSQL (row-oriented) from analytical databases like DuckDB and BigQuery (column-oriented). See the [Apache Arrow docs on the columnar format](https://arrow.apache.org/docs/format/Columnar.html) and the [DuckDB "Fast Analytical Queries" section](https://duckdb.org/why_duckdb.html) for detailed explanations.

## 2. So Is Pandas Row-Oriented and Polars Column-Oriented?

**No.** This is a common misconception. **Both Pandas and Polars store data in columns.** In Pandas, each column is a contiguous NumPy array, grouped by dtype in a structure called the [BlockManager](https://uwekorn.com/2020/05/24/the-one-pandas-internal.html). In Polars, each column is an [Apache Arrow](https://arrow.apache.org/) array. For numeric data, scanning a column is a contiguous memory read in both libraries.

The real differences are in the **execution engine**, not the storage layout:

| | Pandas | Polars |
|---|---|---|
| **Backend** | Python + C/Cython | Rust (no Python overhead in hot paths) |
| **Parallelism** | Single-threaded — blocked by Python's [GIL](https://docs.python.org/3/glossary.html#term-GIL) | Auto-parallelizes across all CPU cores |
| **Execution** | Eager — each operation runs immediately | Eager + Lazy — can build a query plan, then optimize before running |
| **Query optimization** | None | Predicate pushdown, projection pruning, common subexpression elimination |
| **String storage** | Python objects scattered across the heap (pointer chasing, ~50+ bytes overhead per string) | Arrow contiguous byte buffers (cache-friendly, minimal overhead) |
| **API style** | Index-based — `.iloc[]`, `.iterrows()`, `.apply(axis=1)` | Expression-based — `pl.col()`, `pl.when()`, `.group_by().agg()` |

The performance gap comes primarily from: **(1)** Rust eliminating Python interpreter overhead, **(2)** no GIL enabling parallelism, **(3)** lazy evaluation allowing query optimization, and **(4)** Arrow's string storage eliminating Python object overhead. Wes McKinney (creator of Pandas) documented these limitations in ["Apache Arrow and the '10 Things I Hate About Pandas'"](https://wesmckinney.com/blog/apache-arrow-pandas-internals/) — his complaints are about Python object overhead, memory management (5-10x RAM needed), and the lack of query planning, *not* about column vs row storage.

Sources:
- McKinney, W. (2017). [Apache Arrow and the "10 Things I Hate About Pandas."](https://wesmckinney.com/blog/apache-arrow-pandas-internals/)
- Korn, U. (2020). [The one pandas internal you should know about.](https://uwekorn.com/2020/05/24/the-one-pandas-internal.html) — Pandas BlockManager internals
- [Apache Arrow Columnar Format specification](https://arrow.apache.org/docs/format/Columnar.html)
- [Polars User Guide — Coming from Pandas](https://docs.pola.rs/user-guide/migration/pandas/)
- [DuckDB — Why DuckDB](https://duckdb.org/why_duckdb.html) — columnar-vectorized execution in databases

## 3. Setup

In [ ]:
import polars as pl
import pandas as pd
import time

## 4. Load the Data

Load the `cleaned_dataset.parquet` you saved in Part 1 into **both** Polars and Pandas. We need the same data in both libraries so comparisons are apples-to-apples.

In [ ]:
df_pl = pl.read_parquet("cleaned_dataset.parquet")
df_pd = pd.read_parquet("cleaned_dataset.parquet")

print(f"Polars: {df_pl.shape[0]:,} rows × {df_pl.shape[1]} cols")
print(f"Pandas: {df_pd.shape[0]:,} rows × {df_pd.shape[1]} cols")

### Timing helper

We'll use this throughout to compare execution times. Run this cell.

In [ ]:
def compare(pandas_fn, polars_fn, label="", n_runs=3):
    """Time a pandas and polars operation side-by-side. Returns both results."""
    pd_times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        pd_result = pandas_fn()
        pd_times.append(time.perf_counter() - start)

    pl_times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        pl_result = polars_fn()
        pl_times.append(time.perf_counter() - start)

    pd_avg = sum(pd_times) / n_runs
    pl_avg = sum(pl_times) / n_runs
    faster = "Polars" if pl_avg < pd_avg else "Pandas"
    ratio = max(pd_avg, pl_avg) / min(pd_avg, pl_avg)

    print(f"{'— ' + label + ' —' if label else ''}")
    print(f"  Pandas: {pd_avg:.4f}s  |  Polars: {pl_avg:.4f}s  |  {faster} {ratio:.1f}x faster")
    return pd_result, pl_result

---

## 5. Vectorized Column Operations — Where Polars Pulls Ahead

Both libraries store columns contiguously, so why is Polars faster at column operations? Because Polars processes them in **Rust without the GIL**, can **parallelize across columns**, and can **optimize multi-step queries** before executing. Pandas uses C/Cython for some operations but falls back to Python for many, and the GIL prevents true parallelism.

### 5a. Column aggregation

Compute the mean of every numeric column.

Both libraries scan each column as a contiguous array — the storage is equivalent. The difference is in how the engine dispatches the work.

**Pandas:** `df_pd.mean(numeric_only=True)`
**Polars:** `df_pl.select(pl.col(pl.NUMERIC_DTYPES).mean())`

In [ ]:
# TODO: use compare() to time column-wise mean in both libraries

### 4b. Filtering rows by a column predicate

Filter the dataset to only rows where a specific sensor column exceeds its mean value.

Both libraries scan a single column to evaluate the predicate. The difference: Polars evaluates the predicate in Rust; Pandas goes through NumPy's C layer but with more Python overhead for DataFrame construction.

**Pandas:** `df_pd[df_pd["column"] > df_pd["column"].mean()]`
**Polars:** `df_pl.filter(pl.col("column") > pl.col("column").mean())`

In [ ]:
# TODO: pick a sensor column from your dataset and time the filter operation

### 4c. Group-by aggregation

Group by the label/attack column and compute the mean of all numeric columns per group.

This is where parallelism matters: Polars can aggregate multiple columns simultaneously across threads. Pandas is single-threaded due to the [GIL](https://docs.python.org/3/glossary.html#term-GIL) and processes columns sequentially.

**Pandas:** `df_pd.groupby("label")[numeric_cols].mean()`
**Polars:** `df_pl.group_by("label").agg(pl.col(pl.NUMERIC_DTYPES).mean())`

In [ ]:
# TODO: group-by your label column, compute mean of numeric columns, compare timing

### 4d. Adding a computed column

Create a new column that is the difference between two sensor columns (e.g., `sensor_A - sensor_B`).

This is a vectorized operation in both libraries — NumPy does the arithmetic for Pandas, Rust does it for Polars. Both are fast. Try it and see how close they are.

**Pandas:** `df_pd["diff"] = df_pd["sensor_A"] - df_pd["sensor_B"]`
**Polars:** `df_pl.with_columns((pl.col("sensor_A") - pl.col("sensor_B")).alias("diff"))`

In [ ]:
# TODO: pick two sensor columns and compute their difference in both libraries

---

## 6. Row-Wise Operations — Both Libraries Suffer

When you drop out of vectorized column operations and process rows one at a time, **both libraries are slow** — you're back in Python, calling your function once per row, and neither engine can help you. The difference here is more about API ergonomics than raw speed.

### 6a. Row-by-row iteration

Iterate over the first 10,000 rows and collect each row as a dictionary. This is inherently sequential — no parallelism possible.

**Pandas:** `for _, row in df_pd.head(10000).iterrows(): row.to_dict()`
**Polars:** `for row in df_pl.head(10000).iter_rows(named=True): pass`

Pandas' `.iterrows()` returns a labeled `Series` per row (familiar but allocates a new object each time). Polars' `iter_rows(named=True)` returns plain dicts (less overhead).

In [ ]:
# TODO: iterate over 10,000 rows as dicts in both libraries, compare timing

### 5b. Row-wise apply vs columnar reformulation

Apply a custom Python function across multiple columns per row: flag a row as "anomalous" if sensor_A is above its 95th percentile **and** sensor_B is below its 5th percentile.

Both libraries have to call your Python function once per row — neither can optimize this. The lesson: **if you can reformulate a row-wise operation as column expressions, always do it.** The speedup is dramatic.

**Pandas (slow):** `df_pd.apply(lambda row: row["A"] > threshold_a and row["B"] < threshold_b, axis=1)`
**Polars (slow):** `df_pl.map_rows(lambda row: row[0] > threshold_a and row[1] < threshold_b)`
**Polars (fast):** `df_pl.with_columns(pl.when((pl.col("A") > threshold_a) & (pl.col("B") < threshold_b)).then(True).otherwise(False).alias("anomalous"))`

In [ ]:
# TODO: implement the anomaly flag three ways:
#   1. Pandas .apply(axis=1)
#   2. Polars .map_rows()
#   3. Polars columnar expression (pl.when)
# Time all three. Which is fastest? By how much?

### 5c. Single-row lookup

Look up a single row by its integer position (e.g., row 5000). Fast in both — this is a constant-time index into an array. The difference is in the return type.

**Pandas:** `df_pd.iloc[5000]` — returns a labeled `Series`
**Polars:** `df_pl.row(5000, named=True)` — returns a dict

In [ ]:
# TODO: look up row 5000 in both libraries, compare timing and output format

---

## 7. Lazy Evaluation — Polars' Other Advantage

Beyond memory layout, Polars has a feature Pandas lacks entirely: **lazy evaluation**. Instead of executing each operation immediately, you can build a query plan and let Polars optimize it before running.

This matters because Polars can:
- **Predicate pushdown** — apply filters before reading all columns, skipping unnecessary work
- **Projection pushdown** — only read the columns you actually use
- **Common subexpression elimination** — avoid recomputing the same thing twice

See [Polars Lazy API docs](https://docs.pola.rs/user-guide/lazy/using/).

### 7a. Eager vs lazy comparison

Chain together: filter rows → select a subset of columns → group by label → compute mean. Do this twice in Polars — once eager, once lazy — and compare.

**Polars eager:** `df_pl.filter(...).select(...).group_by(...).agg(...)`
**Polars lazy:** `df_pl.lazy().filter(...).select(...).group_by(...).agg(...).collect()`

In [ ]:
# TODO: build a multi-step query (filter → select → group_by → agg)
# Run it eager and lazy in Polars. Compare timing.

### 7b. Inspect the query plan

Before calling `.collect()`, you can see what Polars plans to do. This is useful for understanding what optimizations are happening.

**Polars:** `query.explain()` (optimized plan) or `query.explain(optimized=False)` (naive plan)

In [ ]:
# TODO: build a lazy query and print both the naive and optimized plans
# What differences do you see?

---

## 8. Summary

Fill in this table with your actual timing results from the exercises above.

| Operation | Pandas time | Polars time | Winner | Why |
|-----------|------------|------------|--------|-----|
| Column mean | | | | |
| Filter by predicate | | | | |
| Group-by aggregation | | | | |
| Column arithmetic | | | | |
| Row iteration (10k rows) | | | | |
| Row-wise apply | | | | |
| Single-row lookup | | | | |
| Lazy vs eager chain | N/A | | | |

## 9. Reflection

Answer in 3-5 sentences: based on what you observed, when would you choose Pandas over Polars? When would you choose Polars? Is there a scenario where it doesn't matter?

*TODO: your reflection here*

---

**Sources referenced in this notebook:**
- McKinney, W. (2017). [Apache Arrow and the "10 Things I Hate About Pandas."](https://wesmckinney.com/blog/apache-arrow-pandas-internals/) — Pandas creator on its architectural limitations
- Korn, U. (2020). [The one pandas internal you should know about.](https://uwekorn.com/2020/05/24/the-one-pandas-internal.html) — Pandas BlockManager internals
- [Apache Arrow Columnar Format specification](https://arrow.apache.org/docs/format/Columnar.html)
- [Polars User Guide](https://docs.pola.rs/user-guide/) — architecture, expressions, lazy API
- [Polars — Coming from Pandas](https://docs.pola.rs/user-guide/migration/pandas/)
- [Pandas API reference](https://pandas.pydata.org/docs/reference/index.html)
- [DuckDB — Why DuckDB](https://duckdb.org/why_duckdb.html) — columnar-vectorized execution in databases
- [Python GIL glossary entry](https://docs.python.org/3/glossary.html#term-GIL)